# 5G O-RAN CTI — Standalone Web UI Demo

Runs the CTI pipeline's **web interface only** — no need to run the training notebook.

**Prerequisites (already on Google Drive under `MyDrive/CTI/`):**
- `Network_Dataset.csv` (the dataset)
- `xgb_model.json` (trained model)
- `qwen25_7b_cti_lora/` (fine-tuned LoRA adapters)

**Runtime:** Colab **T4 GPU**. Run the cells top to bottom; the last cell prints a
`https://…gradio.live` link — open it and record the demo.

In [1]:
# 1. Install all dependencies up front (before torch is imported); Unsloth first.
import subprocess, sys
def _pip(*a): subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', *a])
_pip('xgboost', 'shap', 'pandas', 'numpy', 'scikit-learn', 'imbalanced-learn', 'ollama', 'gradio')
_pip('unsloth', 'trl>=0.8.0', 'peft>=0.10.0', 'accelerate>=0.28.0', 'datasets')
subprocess.run([sys.executable, '-m', 'pip', 'uninstall', '-y', 'torchao'], check=False)  # Unsloth uses bitsandbytes
import unsloth, torch
print('unsloth', unsloth.__version__, '| CUDA:', torch.cuda.is_available())

🦥 Unsloth: Will patch your computer to enable 2x faster free finetuning.
🦥 Unsloth Zoo will now patch everything to make training faster!
unsloth 2026.6.9 | CUDA: True


In [3]:
# 2. Mount Google Drive
from google.colab import drive
drive.mount('/content/drive')
DRIVE = '/content/drive/MyDrive/CTI'
CSV_PATH = f'{DRIVE}/Network_Dataset.csv'

Mounted at /content/drive


In [ ]:
# 3. Get the app files (cti_core.py + demo_app.py) onto the runtime
import os
if not os.path.exists('5g-oran-cti-code-exercise'):
    !git clone -q https://github.com/chira99/5g-oran-cti-code-exercise.git
print('app files:', os.listdir('5g-oran-cti-code-exercise/app'))

In [ ]:
# 4. Install + start Ollama and pull the base model (needed for the Base toggle)
!apt-get install -y zstd lshw -qq
!curl -fsSL https://ollama.com/install.sh | sh
import subprocess, time
subprocess.Popen(['ollama', 'serve'], stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL)
time.sleep(5)
!ollama pull qwen2.5:7b
print('Ollama ready.')

In [ ]:
# 5. Build the demo bundle from the CSV (skipped if it already exists on Drive).
# Reproduces the pipeline's preprocessing exactly: same drops, encoders, and a
# stratified 80/20 split with seed 42 -> identical X_test + encoders.
import os
BUNDLE = f'{DRIVE}/demo_bundle.pkl'
if os.path.exists(BUNDLE):
    print('demo_bundle.pkl already on Drive - skipping rebuild.')
else:
    import pandas as pd, numpy as np, joblib
    from sklearn.preprocessing import LabelEncoder
    from sklearn.model_selection import train_test_split

    RANDOM_STATE = 42
    df = pd.read_csv(CSV_PATH)
    df = df[df['attack_category'] != 'bruteforce'].copy()
    DROP_COLS = ['uid', 'src_ip', 'dst_ip', 'attack_type', 'traffic_type', 'history', 'missed_bytes']
    dfm = df.drop(columns=DROP_COLS)

    feature_encoders = {}
    for col in ['proto', 'service', 'conn_state']:
        le = LabelEncoder(); dfm[col] = le.fit_transform(dfm[col].astype(str)); feature_encoders[col] = le

    target_encoder = LabelEncoder()
    y = target_encoder.fit_transform(dfm['attack_category'])
    CLASS_NAMES = list(target_encoder.classes_)
    X = dfm.drop(columns=['attack_category'])

    X_tr, X_test, y_tr, y_test = train_test_split(
        X, y, test_size=0.2, stratify=y, random_state=RANDOM_STATE)

    N_PER_CLASS = 5
    rng = np.random.RandomState(RANDOM_STATE)
    idx = []
    for c in range(len(CLASS_NAMES)):
        pool = np.where(y_test == c)[0]
        idx += rng.choice(pool, size=min(N_PER_CLASS, len(pool)), replace=False).tolist()
    events = X_test.iloc[idx].copy()
    events['_true_label'] = [CLASS_NAMES[int(y_test[i])] for i in idx]
    events['_orig_idx'] = idx

    joblib.dump({'feature_encoders': feature_encoders, 'class_names': CLASS_NAMES,
                 'feature_columns': list(X.columns), 'events': events}, BUNDLE)
    print('Saved demo_bundle.pkl |', events['_true_label'].value_counts().to_dict())

In [ ]:
# 6. Launch the web UI. Open the printed https://...gradio.live link.
# (This cell stays running while the app is live; press Stop to end.)
!python 5g-oran-cti-code-exercise/app/demo_app.py